In [1]:
!pip install openai

In [2]:
from openai import OpenAI

In [3]:
model = OpenAI(
    api_key="",
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
r1 = model.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": "What is AQI of ahemdabad city, India"
            }
        ],
        max_completion_tokens=50,
        temperature=0
    )

In [8]:
r1

ChatCompletion(id='chatcmpl-29579913-81b5-4648-944d-7d68570acc7a', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content="I'm not aware of the current AQI (Air Quality Index) of Ahmedabad city, India. However, I can suggest some ways to find the current AQI:\n\n1. **Check online AQI websites**: You can check websites like Air", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774171140, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_4387d3edbb', usage=CompletionUsage(completion_tokens=50, prompt_tokens=46, total_tokens=96, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.04640638, prompt_time=0.00214536, completion_time=0.09300181, total_time=0.09514717), usage_breakdown=None, x_groq={'id': 'req_01kmadeca7f0qs4d4fwvgeb8ae', 'seed': 311287554})

In [12]:
import json

# Convert the response to a JSON string and then pretty-print it
json_string = r1.to_json()
parsed_json = json.loads(json_string)
print(json.dumps(parsed_json, indent=2))

{
  "id": "chatcmpl-29579913-81b5-4648-944d-7d68570acc7a",
  "choices": [
    {
      "finish_reason": "length",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "I'm not aware of the current AQI (Air Quality Index) of Ahmedabad city, India. However, I can suggest some ways to find the current AQI:\n\n1. **Check online AQI websites**: You can check websites like Air",
        "role": "assistant"
      }
    }
  ],
  "created": 1774171140,
  "model": "llama-3.1-8b-instant",
  "object": "chat.completion",
  "service_tier": "on_demand",
  "system_fingerprint": "fp_4387d3edbb",
  "usage": {
    "completion_tokens": 50,
    "prompt_tokens": 46,
    "total_tokens": 96,
    "queue_time": 0.04640638,
    "prompt_time": 0.00214536,
    "completion_time": 0.09300181,
    "total_time": 0.09514717
  },
  "usage_breakdown": null,
  "x_groq": {
    "id": "req_01kmadeca7f0qs4d4fwvgeb8ae",
    "seed": 311287554
  }
}


In [13]:
r1.choices[0].message.content

"I'm not aware of the current AQI (Air Quality Index) of Ahmedabad city, India. However, I can suggest some ways to find the current AQI:\n\n1. **Check online AQI websites**: You can check websites like Air"

In [14]:
# this is just a JSON, we don't need to write function right now
# 3 things to assign here, name, description, parameters(nested object with properties)
function_description = [
    {
        "name": "get_aqi_info",
        "description": "Get the AQI info of a city",
        "parameters":{
            # this requires input parameters
            "type":"object",
            "properties": {
                "city" :{
                    "type":"string",
                    "description":"name of the city"
                },
                "country":{
                    "type":"string",
                    "description":"name of the country"
                },
            },
            "required":["city","country"]
        }
    }
]

In [15]:
user_prompt = "What is the AQI of Ahemdabad City of India"

In [31]:
r2 = model.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    functions = function_description,
    function_call="auto"

)

In [27]:
print(r2.choices[0].message.function_call)

FunctionCall(arguments='{"city":"Ahemedabad","country":"India"}', name='get_aqi_info')


In [32]:
r22 = json.dumps(json.loads(r2.to_json()), indent=2)

In [33]:
print(r22)

{
  "id": "chatcmpl-a19d6ddc-69ca-4ac0-96ae-f6dc08139ecb",
  "choices": [
    {
      "finish_reason": "function_call",
      "index": 0,
      "logprobs": null,
      "message": {
        "role": "assistant",
        "function_call": {
          "arguments": "{\"city\":\"Ahemdabad\",\"country\":\"India\"}",
          "name": "get_aqi_info"
        }
      }
    }
  ],
  "created": 1774172558,
  "model": "llama-3.1-8b-instant",
  "object": "chat.completion",
  "service_tier": "on_demand",
  "system_fingerprint": "fp_4387d3edbb",
  "usage": {
    "completion_tokens": 24,
    "prompt_tokens": 245,
    "total_tokens": 269,
    "queue_time": 0.04536322,
    "prompt_time": 0.01408546,
    "completion_time": 0.033590057,
    "total_time": 0.047675517
  },
  "usage_breakdown": null,
  "x_groq": {
    "id": "req_01kmaesn8sfcfv9ktt5reah4ha",
    "seed": 416418634
  }
}


In [55]:
# here we got arguments
def get_aqi_info(city, country):
    """ Get Air Quality Index (AQI) for a city in a country"""

    # let's hardcode this response as of now
    aqi_info = {
        "city": city,
        "country": country,
        "aqi": 100,
        "aqi_description": "Good"
    }

    return json.dumps(aqi_info)




In [52]:
r2.choices[0].message.function_call

FunctionCall(arguments='{"city":"Ahemdabad","country":"India"}', name='get_aqi_info')

In [57]:
import json # Ensure json is imported for parsing function arguments

# Extract arguments from the function call suggested by the model in r2
function_call_args = json.loads(r2.choices[0].message.function_call.arguments)

second_completion = model.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages = [
        {"role": "user", "content": user_prompt},
        r2.choices[0].message, # Include the assistant's message that suggested the function call
        {
            "role": "function",
            "name": r2.choices[0].message.function_call.name,
            "content": get_aqi_info(function_call_args["city"], function_call_args["country"]),
        }
    ],
    functions = function_description,
    function_call="auto"
)

In [58]:
second_completion

ChatCompletion(id='chatcmpl-7e28fc18-e0f0-4ecf-9797-05b983b376d8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The AQI of Ahemdabad City in India is 100, which corresponds to a "Good" description.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774173593, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_f757f4b0bf', usage=CompletionUsage(completion_tokens=24, prompt_tokens=305, total_tokens=329, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.046870179, prompt_time=0.05303792, completion_time=0.037028041, total_time=0.090065961), usage_breakdown=None, x_groq={'id': 'req_01kmafs7g2e67r15twkzhfptzt', 'seed': 1297774995})

In [ ]:
## let's define 3 functions

In [61]:
import datetime

In [59]:
def get_weather(city: str) -> str:
    """ This function will help in getting weather of a city """
    return f'it is 30 degree celcius in {city}'

In [60]:
def calculate(expression: str)-> str:
    """ This function will calculate an expression """
    try:
        r = eval(expression, {"__builtins__":{}},{})
        return str(r)
    except Exception as e:
        return f"Error: {e}"

In [62]:
def get_current_time() -> str:
    """ This function will give current time """
    return f"Current time is {datetime.datetime.now()}"

In [ ]:
## tools schema

In [114]:
TOOLS = [
    {
        "type": "function",
        "function" : {
            "name" : "get_weather",
            "description" : "blah blah blah blah blah",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function" : {
            "name": "calculate",
            "description": "This function will calculate an expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "expression to be calculated"
                    }
                },
                "required": ["expression"]
            }
            }
        },
    {
        "type": "function",
        "function" : {
            "name": "get_current_time",
            "description": "This function will give current time",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }

]

In [64]:
## Tool registry -> maps tool names to actual function
tool_registry = {
    "get_weather": get_weather,
    "calculate": calculate,
    "get_current_time": get_current_time
}

In [68]:
## let's see how description affects model response
r3 = model.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages = [
        {
            "role": "user",
            "content": "what is the temperature in Delhi?"
        }
    ],
    tools = TOOLS,
    tool_choice = "auto"

)

In [69]:
r3

ChatCompletion(id='chatcmpl-c0bea4e3-7069-464a-84fb-d46ceea46569', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='69e62j0yw', function=Function(arguments='{"city":"Delhi"}', name='get_weather'), type='function')]))], created=1774180863, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_4387d3edbb', usage=CompletionUsage(completion_tokens=15, prompt_tokens=337, total_tokens=352, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.046432625, prompt_time=0.028413224, completion_time=0.028704007, total_time=0.057117231), usage_breakdown=None, x_groq={'id': 'req_01kmapq3n1e1eak5txnw1ry3h2', 'seed': 1924307713})

In [71]:
# so, now i am supposed to check if model wants to call tool or it want to end
r3.choices[0].finish_reason

'tool_calls'

In [80]:
# what tool to call
r3.choices[0].message.tool_calls[0].function.name

'get_weather'

In [92]:
r3.choices[0].message.tool_calls[0].function.arguments

'{"city":"Delhi"}'

In [94]:
a = json.loads(r3.choices[0].message.tool_calls[0].function.arguments)
a

{'city': 'Delhi'}

In [81]:
tool_registry['get_weather']

<function __main__.get_weather(city: str) -> str>

In [96]:
if r3.choices[0].finish_reason == "tool_calls":
    print("Model wants to call a tool")
    # model wants to call
    fin_name = r3.choices[0].message.tool_calls[0].function.name
    print(f"model wants to call", fin_name)
    my_fun = tool_registry[fin_name]
    # executing my function
    a = json.loads(r3.choices[0].message.tool_calls[0].function.arguments)
    res = my_fun(a['city'])





Model wants to call a tool
model wants to call get_weather


In [97]:
print(res)

it is 30 degree celcius in Delhi


In [98]:
tool_call = r3.choices[0].message.tool_calls[0]

In [100]:
tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)

In [ ]:
# now giving it to llm

In [107]:
final_call = model.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages = [
        {
                "role": "user",
                "content": "what is the temperature in Delhi?"
            },
        r3.choices[0].message,
        {
                "role": "tool",
                "tool_call_id": r3.choices[0].message.tool_calls[0].id,   # must match the tool call id
                "content": str(res)           # your function's return value
            }
    ],
    tools = TOOLS,
    tool_choice="none"
)

In [102]:
print(final_call)

ChatCompletion(id='chatcmpl-801a2d30-ca76-4909-bb73-81e1ff3c16c0', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='nv91mv263', function=Function(arguments='{"city":"Delhi"}', name='get_weather'), type='function')]))], created=1774182414, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_f757f4b0bf', usage=CompletionUsage(completion_tokens=21, prompt_tokens=371, total_tokens=392, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.045997017, prompt_time=0.023624852, completion_time=0.019867729, total_time=0.043492581), usage_breakdown=None, x_groq={'id': 'req_01kmar6ef1e4pbam97n2pnbzyw', 'seed': 1735121771})


In [106]:
print(final_call.choices[0].message.content)

None


In [108]:
final_call

ChatCompletion(id='chatcmpl-7fc67fca-f2c9-47fc-9ce3-63d9bdf4ba2f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Please note that the temperature I provided is a prediction and may vary based on actual weather conditions.\n\nAlso, I must correct myself, I'm not supposed to use functions as per your request. Here's a direct answer:\n\nThe temperature in Delhi can vary depending on the time of year and other factors, but as of my knowledge cutoff in December 2023, the temperature in Delhi can range from 10°C to 40°C (50°F to 104°F).", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774182848, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_f757f4b0bf', usage=CompletionUsage(completion_tokens=95, prompt_tokens=103, total_tokens=198, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.04570301, pr

In [109]:
messages = [
        {
                "role": "user",
                "content": "what is the temperature in Delhi?"
            },
        r3.choices[0].message,
        {
                "role": "tool",
                "tool_call_id": r3.choices[0].message.tool_calls[0].id,   # must match the tool call id
                "content": str(res)           # your function's return value
            }
    ]
print(messages)

[{'role': 'user', 'content': 'what is the temperature in Delhi?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='69e62j0yw', function=Function(arguments='{"city":"Delhi"}', name='get_weather'), type='function')]), {'role': 'tool', 'tool_call_id': '69e62j0yw', 'content': 'it is 30 degree celcius in Delhi'}]


In [115]:
def run_with_tools(user_message: str):
    messages = [{"role": "user", "content": user_message}]

    print(f"\nUser: {user_message}")
    print("-" * 50)

    while True:
        # Step 1: send messages + tools to the model
        response = model.chat.completions.create(
            model= "llama-3.1-8b-instant",
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        # Step 2: add model's reply to history (always do this)
        messages.append(msg)

        # Step 3: check if model wants to call a tool
        if finish_reason == "tool_calls":
            print(f"Model wants to call: {[tc.function.name for tc in msg.tool_calls]}")

            # Step 4: execute each tool the model requested
            for tool_call in msg.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                print(f"  Running {tool_name}({tool_args})")

                # run the actual Python function
                try:
                    tool_fn = tool_registry[tool_name]
                    tool_result = tool_fn(**tool_args)
                except Exception as e:
                    tool_result = f"Error running tool: {e}"

                print(f"  Result: {tool_result}")

                # Step 5: add tool result to messages
                # notice: role is "tool", not "user" or "assistant"
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,  # must match the request id
                    "content": tool_result
                })

            # loop back → model now has tool results, will give final answer

        else:
            # Step 6: no tool call → this is the final answer
            print(f"\nFinal answer: {msg.content}")
            return msg.content


In [111]:
# test 1: needs a tool
run_with_tools("What is the weather in Surat?")



User: What is the weather in Surat?
--------------------------------------------------
Model wants to call: ['get_weather']
  Running get_weather({'city': 'Surat'})
  Result: it is 30 degree celcius in Surat

Final answer: I can help with the weather but I don't have access to real-time weather data.


"I can help with the weather but I don't have access to real-time weather data."

In [112]:
# test 2: needs NO tool — model should answer directly
run_with_tools("What is the capital of France?")



User: What is the capital of France?
--------------------------------------------------

Final answer: <brave_search> {"query": "capital of France"}


'<brave_search> {"query": "capital of France"}'

In [113]:
# test 3: needs the calculator tool
run_with_tools("What is 1234 multiplied by 5.67?")


User: What is 1234 multiplied by 5.67?
--------------------------------------------------
Model wants to call: ['calculate']
  Running calculate({'expression': '1234 * 5.67'})
  Result: 6996.78
Model wants to call: ['calculate']
  Running calculate({'expression': '1234 * 5.67'})
  Result: 6996.78

Final answer: However, we can simplify the calculation by converting 1234 into an integer and then calculating.


'However, we can simplify the calculation by converting 1234 into an integer and then calculating.'

In [116]:
# checking if bad description works
# test 4: needs a tool
run_with_tools("What is the weather in Surat?")



User: What is the weather in Surat?
--------------------------------------------------
Model wants to call: ['get_weather']
  Running get_weather({'city': 'Surat'})
  Result: it is 30 degree celcius in Surat

Final answer: Note: The final response is a mock one and doesn't have real time information as the get_weather function was not actually called.


"Note: The final response is a mock one and doesn't have real time information as the get_weather function was not actually called."

In [117]:
# test 4: needs datetime tool
run_with_tools("What day and time is it right now?")


User: What day and time is it right now?
--------------------------------------------------
Model wants to call: ['get_current_time']
  Running get_current_time(None)
  Result: Error running tool: __main__.get_current_time() argument after ** must be a mapping, not NoneType

Final answer: No data found from <function=get_current_time>null</function>.


'No data found from <function=get_current_time>null</function>.'

In [124]:
messages_debug = [{"role": "user", "content": "What is the weather in Mumbai?"}]

response = model.chat.completions.create(
    model="llama-3.1-8b-instant", messages=messages_debug, tools=TOOLS
)

In [119]:
msg = response.choices[0].message
messages_debug.append(msg)

In [120]:
# look at what the model sent back — a tool_call object
print("=== Model's raw reply (tool_call) ===")
print(f"finish_reason : {response.choices[0].finish_reason}")
print(f"tool name     : {msg.tool_calls[0].function.name}")
print(f"tool args     : {msg.tool_calls[0].function.arguments}")
print(f"tool_call_id  : {msg.tool_calls[0].id}")

=== Model's raw reply (tool_call) ===
finish_reason : tool_calls
tool name     : get_weather
tool args     : {"city":"Mumbai"}
tool_call_id  : c40qrnbd4


In [121]:
messages_debug.append({
    "role": "tool",
    "tool_call_id": msg.tool_calls[0].id,
    "content": get_weather("Mumbai")
})

In [125]:
# messages_debug = [{"role": "user", "content": "What is the weather in Mumbai?"}]

response2 = model.chat.completions.create(
    model="llama-3.1-8b-instant", messages=messages_debug, tools=TOOLS
)

In [126]:
print("\n=== Final answer ===")
print(response2.choices[0].message.content)


=== Final answer ===
None


In [127]:
run_with_tools('What is 1/0 ?')


User: What is 1/0 ?
--------------------------------------------------
Model wants to call: ['calculate']
  Running calculate({'expression': '1/0'})
  Result: Error: division by zero
Model wants to call: ['calculate']
  Running calculate({'expression': 'try: 1/0 except ZeroDivisionError: 0'})
  Result: Error: invalid syntax (<string>, line 1)
Model wants to call: ['calculate']
  Running calculate({'expression': '1.0/0'})
  Result: Error: float division by zero
Model wants to call: ['calculate']
  Running calculate({'expression': 'math.isclose(1/0, -inf, rel_tol = -1e-6) or math.isclose(1/0, inf, abs_tol = 1e-6)'})
  Result: Error: name 'math' is not defined

Final answer: To answer what is 1/0, we can't say it's a specific number because division by zero is undefined.


"To answer what is 1/0, we can't say it's a specific number because division by zero is undefined."

In [128]:
run_with_tools('What is 3+4 and what is weather in Delhi?')


User: What is 3+4 and what is weather in Delhi?
--------------------------------------------------
Model wants to call: ['calculate', 'get_weather']
  Running calculate({'expression': '3+4'})
  Result: 7
  Running get_weather({'city': 'Delhi'})
  Result: it is 30 degree celcius in Delhi

Final answer: Note: Since the actual temperature in Delhi is not provided, it's mentioned as per the get_weather function. In reality, this function does not have an output as mentioned above, it should return the weather details of the city like temperature, humidity etc. 

Also, to provide a final answer in the format requested, the final answer would look like:
 The final answer is $\boxed{7}$.


"Note: Since the actual temperature in Delhi is not provided, it's mentioned as per the get_weather function. In reality, this function does not have an output as mentioned above, it should return the weather details of the city like temperature, humidity etc. \n\nAlso, to provide a final answer in the format requested, the final answer would look like:\n The final answer is $\\boxed{7}$."